# Week 7 Implementation: ES vs Rank-1 LoRA-ES on GridWorld

**Date:** February 24, 2026  
**Course:** STAT 4830

This notebook compares standard Evolution Strategies (ES) against rank-1 LoRA ES (LoRA-only parameter updates) on sparse-reward GridWorld. We run both methods with all supported perturbation noise types: Gaussian, Cauchy, and Laplace.

## Problem Setup

### Clear Problem Statement

**Goal:** Learn a policy $\pi_\theta$ that reaches the goal in sparse-reward GridWorld.

**Challenge:** Sparse rewards and non-differentiable environment interactions make direct gradient optimization difficult.

**Approach:** Compare parameter-space ES on:
1. **Standard policy parameters** (`param_mode='all'`)
2. **Rank-1 LoRA-only parameters** (`param_mode='lora'`, base frozen)

Each method is evaluated under **all three ES noise distributions**.

### Mathematical Formulation

We optimize expected return:
$$
\max_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} r_t\right]
$$

ES gradient estimator:
$$
\nabla_\theta J(\theta) \approx \frac{1}{N\sigma}\sum_{i=1}^{N} F_i\, s(\epsilon_i)
$$
where $s(\epsilon)$ is the score-function term for the selected noise distribution.

For rank-1 LoRA on each linear layer:
$$
W' = W + \alpha\, b a^\top
$$
with vector factors $a$ and $b$ (rank = 1). In LoRA-only mode, ES updates only adapter vectors while $W$ is frozen.

### Data Requirements

**Environment:**
- Grid size: 8x8
- Obstacles: 8
- State space: one-hot position (64 dims)
- Action space: 4 actions
- Max steps per episode: 50

**Reward scheme (Week 4-style):**
- **Training:** distance-based shaped rewards (dense signal)
- **Evaluation:** original sparse rewards (0/+1 with obstacle penalty)

**Training Configuration (default in this notebook):**
- Iterations: 80
- Population size: 50
- Sigma: 0.10
- Alpha: 0.05
- Episodes per perturbation: 5
- Noise types: Gaussian, Cauchy, Laplace

### Success Metrics

1. **Final Evaluation Reward**
2. **Final Success Rate**
3. **Gradient Norm Trend**
4. **Runtime per run**
5. **Parameter efficiency** (trainable parameters in ES search space)

## Implementation

In [7]:
# All required imports
import sys
sys.path.append('../src')

import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from model import GridWorld, PolicyNetwork
from utils import NOISE_TYPES, train_es, evaluate_policy

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

print('Imports successful!')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
print(f'Noise types: {NOISE_TYPES}')

Imports successful!
PyTorch version: 2.10.0+cpu
Device: cpu
Noise types: ('gaussian', 'cauchy', 'laplace')


### Environment Implementation

In [8]:
# Distance-based reward shaping (same strategy as Week 4)
class ShapedRewardEnvComparison(GridWorld):
    """GridWorld with distance-based reward shaping."""

    def reset(self):
        state = super().reset()
        self.prev_dist = abs(self.agent_pos[0] - self.goal_pos[0]) + abs(self.agent_pos[1] - self.goal_pos[1])
        return state

    def step(self, action):
        state, reward, done, info = super().step(action)
        curr_dist = abs(self.agent_pos[0] - self.goal_pos[0]) + abs(self.agent_pos[1] - self.goal_pos[1])
        shaped_reward = reward + 0.2 * (self.prev_dist - curr_dist) - 0.01
        self.prev_dist = curr_dist
        return state, shaped_reward, done, info


# Create training (shaped) and evaluation (sparse) environments
train_env = ShapedRewardEnvComparison(size=8, n_obstacles=8, max_steps=50, seed=123)
eval_env = GridWorld(size=8, n_obstacles=8, max_steps=50, seed=123)

state = train_env.reset()
print(f'Training environment: {train_env.size}x{train_env.size} grid (shaped rewards)')
print(f'State shape: {state.shape}')
print(f'Action space: {train_env.n_actions}')
print(f'Start: {train_env.start_pos} | Goal: {train_env.goal_pos} | Obstacles: {len(train_env.obstacles)}')

train_env.render()

Training environment: 8x8 grid (shaped rewards)
State shape: (64,)
Action space: 4
Start: (7, 0) | Goal: (0, 7) | Obstacles: 8


<Figure size 800x800 with 1 Axes>

### Policy Setup: Standard vs Rank-1 LoRA

In [9]:
state_dim = train_env._get_state().shape[0]
action_dim = train_env.n_actions

standard_policy = PolicyNetwork(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dim=64,
    n_layers=2,
    use_lora=False
).to(device)

lora_policy = PolicyNetwork(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dim=64,
    n_layers=2,
    use_lora=True,
    lora_alpha=1.0,
    lora_init_scale=1e-3
).to(device)

total_standard = sum(p.numel() for p in standard_policy.parameters())
total_lora_model = sum(p.numel() for p in lora_policy.parameters())
lora_trainable = lora_policy.count_lora_parameters()

print('Standard ES policy parameters:')
print(f'  Total parameters searched: {total_standard}')

print('\nLoRA policy parameters:')
print(f'  Total model parameters: {total_lora_model}')
print(f'  LoRA search parameters (rank-1 only): {lora_trainable}')
print(f'  Compression factor in ES search space: {total_lora_model / lora_trainable:.2f}x')

Standard ES policy parameters:
  Total parameters searched: 8580

LoRA policy parameters:
  Total model parameters: 8904
  LoRA search parameters (rank-1 only): 324
  Compression factor in ES search space: 27.48x


### ES Experiment Utilities

In [10]:
def make_policy(use_lora: bool):
    return PolicyNetwork(
        state_dim=state_dim,
        action_dim=action_dim,
        hidden_dim=64,
        n_layers=2,
        use_lora=use_lora,
        lora_alpha=1.0,
        lora_init_scale=1e-3
    ).to(device)


def run_es_variant(name: str, use_lora: bool, param_mode: str, noise_type: str,
                   n_iterations: int = 80, N: int = 50, sigma: float = 0.10,
                   alpha: float = 0.05, n_eval_episodes: int = 5, max_steps: int = 50):
    policy = make_policy(use_lora=use_lora)

    t0 = time.time()
    history = train_es(
        policy=policy,
        env=train_env,
        N=N,
        sigma=sigma,
        alpha=alpha,
        n_iterations=n_iterations,
        n_eval_episodes=n_eval_episodes,
        max_steps=max_steps,
        eval_every=5,
        verbose=True,
        noise_type=noise_type,
        param_mode=param_mode
    )
    elapsed = time.time() - t0

    eval_reward, eval_success, eval_steps = evaluate_policy(
        policy=policy,
        env=eval_env,
        n_episodes=30,
        max_steps=max_steps,
        deterministic=True
    )

    return {
        'method': name,
        'noise': noise_type,
        'param_mode': param_mode,
        'final_eval_reward': float(eval_reward),
        'final_eval_success': float(eval_success),
        'final_eval_steps': float(eval_steps),
        'final_train_fitness': float(history['avg_fitness'][-1]),
        'final_grad_norm': float(history['gradient_norm'][-1]),
        'runtime_sec': float(elapsed),
        'history': history
    }


print('Utility functions ready.')

Utility functions ready.


### Run Experiments (All Noise Types)

This runs 6 configurations total:
- Standard ES with Gaussian/Cauchy/Laplace
- Rank-1 LoRA ES with Gaussian/Cauchy/Laplace

In [11]:
results = []

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"Training device: {device} ({device_name})")

for noise in NOISE_TYPES:
    print(f'Running standard ES with {noise} noise...')
    res_std = run_es_variant(
        name='ES-standard',
        use_lora=False,
        param_mode='all',
        noise_type=noise
    )
    results.append(res_std)

    print(f'Running LoRA ES with {noise} noise...')
    res_lora = run_es_variant(
        name='ES-LoRA-rank1',
        use_lora=True,
        param_mode='lora',
        noise_type=noise
    )
    results.append(res_lora)

print('\nCompleted all runs.')

print('\nSummary table:')
header = f"{'Method':<14} {'Noise':<9} {'Reward':>8} {'Success':>9} {'GradNorm':>10} {'Time(s)':>9}"
print(header)
print('-' * len(header))
for r in results:
    print(f"{r['method']:<14} {r['noise']:<9} {r['final_eval_reward']:>8.3f} {r['final_eval_success']:>9.1%} {r['final_grad_norm']:>10.3f} {r['runtime_sec']:>9.1f}")

Training device: cpu (CPU)
Running standard ES with gaussian noise...
Iter    0 | Fitness: -0.168 | Eval Reward:  0.700 | Success: 0.00% | Grad Norm: 129.9109
Iter    5 | Fitness:  2.164 | Eval Reward: -5.200 | Success: 0.00% | Grad Norm: 130.4106
Iter   10 | Fitness:  2.110 | Eval Reward: -5.200 | Success: 0.00% | Grad Norm: 130.2386
Iter   15 | Fitness:  1.556 | Eval Reward: -3.400 | Success: 0.00% | Grad Norm: 130.2591
Iter   20 | Fitness:  1.845 | Eval Reward:  3.660 | Success: 100.00% | Grad Norm: 128.7572
Iter   25 | Fitness:  2.255 | Eval Reward:  0.900 | Success: 0.00% | Grad Norm: 129.8948
Iter   30 | Fitness:  2.944 | Eval Reward: -0.100 | Success: 0.00% | Grad Norm: 129.1423
Iter   35 | Fitness:  2.990 | Eval Reward: -4.000 | Success: 0.00% | Grad Norm: 130.2975
Iter   40 | Fitness:  2.881 | Eval Reward:  3.660 | Success: 100.00% | Grad Norm: 129.7595
Iter   45 | Fitness:  3.022 | Eval Reward:  1.500 | Success: 0.00% | Grad Norm: 129.4609
Iter   50 | Fitness:  2.738 | Eval R

KeyboardInterrupt: 

### Visual Comparison

In [ ]:
# Build arrays for plotting
labels = [f"{r['method']}\n{r['noise']}" for r in results]
rewards = [r['final_eval_reward'] for r in results]
successes = [r['final_eval_success'] for r in results]
runtimes = [r['runtime_sec'] for r in results]

x = np.arange(len(results))
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].bar(x, rewards, color=['tab:blue' if 'standard' in l else 'tab:orange' for l in labels], alpha=0.8)
axes[0].set_title('Final Evaluation Reward')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=35, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x, successes, color=['tab:blue' if 'standard' in l else 'tab:orange' for l in labels], alpha=0.8)
axes[1].set_title('Final Success Rate')
axes[1].set_ylim(0, 1.0)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=35, ha='right')
axes[1].grid(True, alpha=0.3, axis='y')

axes[2].bar(x, runtimes, color=['tab:blue' if 'standard' in l else 'tab:orange' for l in labels], alpha=0.8)
axes[2].set_title('Runtime (seconds)')
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, rotation=35, ha='right')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Plot training curves by noise (shaped reward on training env)
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
for j, noise in enumerate(NOISE_TYPES):
    ax = axes[j]
    for r in results:
        if r['noise'] != noise:
            continue
        h = r['history']
        ax.plot(h['iteration'], h['eval_reward'], marker='o', linewidth=2,
                label=r['method'])
    ax.set_title(f'Noise: {noise}')
    ax.set_xlabel('Iteration')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Shaped evaluation reward (training env)')
axes[-1].legend(loc='lower right')
plt.tight_layout()
plt.show()

## Week 7 Summary

- Rank-1 LoRA ES reduces ES search dimensionality by optimizing only adapter vectors.
- Standard ES and LoRA ES can respond differently to perturbation distribution choice.
- Running all three noise types helps identify whether performance differences are caused by parameterization (LoRA vs full) or exploration distribution.

If needed, next steps are:
1. Repeat each configuration over multiple random seeds.
2. Tune `sigma` and `alpha` separately for LoRA and standard ES.
3. Extend this exact notebook structure to HarderGridWorld.